#**This code is used to train a Stable Diffusion v1 model with LoRA using 10,000 images and their corresponding captions.**

In [ ]:
!pip install -q \
  "diffusers==0.25.0" \
  "accelerate==0.27.0" \
  "transformers==4.38.0" \
  "peft==0.9.0" \
  "safetensors==0.4.3" \
  "huggingface_hub==0.21.0" \
  --force-reinstall

print("✅ Bitti! Runtime > Oturumu Yeniden Başlat yap, sonra Cell 2'den devam et.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 11.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 279.7/279.7 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.9/190.9 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.1/346.1 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.6/202.6 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_DIR = '/content/drive/MyDrive/celeba_lora'
TRAIN_DIR   = os.path.join(PROJECT_DIR, 'train_data_HQ')
OUTPUT_DIR  = os.path.join(PROJECT_DIR, 'output_hq_10k_v2')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("✅ Hazır!")
print(f"  Train : {TRAIN_DIR}")
print(f"  Output: {OUTPUT_DIR}")

Mounted at /content/drive
✅ Hazır!
  Train : /content/drive/MyDrive/celeba_lora/train_data_HQ
  Output: /content/drive/MyDrive/celeba_lora/output_hq_10k_v2


In [ ]:
!git clone https://github.com/kohya-ss/sd-scripts /content/sd-scripts -q
%cd /content/sd-scripts
!pip install -q -r requirements.txt

# torchvision CUDA uyumsuzluğunu düzelt
!pip install -q --upgrade torchvision --extra-index-url https://download.pytorch.org/whl/cu118

print("✅ Kurulum tamam!")

/content/sd-scripts
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 3.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.7/354.7 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 MB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.4/287.4 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.8/434.8 kB 38.7 MB/s eta 0:00:00
   ━━━━━━━

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

CUDA: True
GPU: Tesla T4


In [ ]:
!pip install -q huggingface_hub

from huggingface_hub import hf_hub_download
import os

MODEL_PATH = '/content/v1-5-pruned-emaonly.safetensors'

if not os.path.exists(MODEL_PATH):
    print("İndiriliyor (~4 GB)...")
    hf_hub_download(
        repo_id="runwayml/stable-diffusion-v1-5",
        filename="v1-5-pruned-emaonly.safetensors",
        local_dir="/content"
    )

print("✅ Model hazır!")

İndiriliyor (~4 GB)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


v1-5-pruned-emaonly.safetensors:   0%|          | 0.00/4.27G [00:00<?, ?B/s]

✅ Model hazır!


In [ ]:
config = f"""
[general]
enable_bucket = true

[[datasets]]
resolution = 512
batch_size = 2

  [[datasets.subsets]]
  image_dir = "{TRAIN_DIR}/10_face"
  caption_extension = ".txt"
  num_repeats = 1
"""

config_path = '/content/dataset_config.toml'
with open(config_path, 'w') as f:
    f.write(config)

print("✅ Config yazıldı!")

✅ Config yazıldı!


In [ ]:
import subprocess

cmd = [
    "accelerate", "launch",
    "/content/sd-scripts/train_network.py",
    f"--pretrained_model_name_or_path={MODEL_PATH}",
    f"--dataset_config={config_path}",
    f"--output_dir={OUTPUT_DIR}",
    "--output_name=celeba_hq_lora",
    "--network_module=networks.lora",
    "--network_dim=32",
    "--network_alpha=16",
    "--optimizer_type=AdamW",
    "--learning_rate=1e-4",
    "--lr_scheduler=cosine_with_restarts",
    "--max_train_steps=10000",
    "--save_every_n_steps=2000",
    "--save_model_as=safetensors",
    "--mixed_precision=fp16",
    "--gradient_checkpointing",
    "--train_batch_size=1",
]

print("🚀 Training başlıyor!")
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end='')
process.wait()
print("\n✅ Training bitti!")

Görüntülenen çıkış son 5000 satıra kısaltıldı.
steps:  87%|████████▋ | 8651/10000 [3:28:26<32:30,  1.45s/it, avr_loss=0.144]


In [ ]:
import glob
import os
checkpoints = sorted(glob.glob(os.path.join(OUTPUT_DIR, '*.safetensors')))
print(f"📦 {len(checkpoints)} checkpoint bulundu:\n")
for i, ckpt in enumerate(checkpoints):
    size_mb = os.path.getsize(ckpt) / 1024 / 1024
    print(f"  [{i}] {os.path.basename(ckpt)}  ({size_mb:.0f} MB)")

In [ ]:
# Önce bunu çalıştır, sonra Runtime > Oturumu Yeniden Başlat yap
!pip install -q --upgrade Pillow
!pip install -q "diffusers==0.25.0" "transformers==4.38.0" "peft==0.9.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 74.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 101.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.9/190.9 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 110.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sente